# The N-Ball Transformer — Data

**No external datasets.** Every value is computed from
`V(n) = π^(n/2)/Γ(n/2+1)`.

| Quantity | Produced by | Arithmetic |
|---|---|---|
| Step ratios, recurrence | `sympy` | exact symbolic |
| `V(n)` reference values | `sympy`, then rounded once | exact → float64 |
| Engine values | `fixed_point.v_nball` | float64, `math.gamma` |
| `n*` | `scipy.optimize.brentq` on `ψ(n/2+1) − ln π` | float64, bracketed |
| Engine `n*` | `fixed_point.v_nball_peak` | float64, 10k-point grid |

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../..'))

from ValaQuenta import fixed_point as fp

import math
import numpy as np
import sympy as sy
import scipy.special as ss
import scipy.optimize as so

print('engine :', 'ValaQuenta/fixed_point.py')
print('python :', sys.version.split()[0])

In [ ]:
# Exact V(n) in rational/symbolic arithmetic -- no floating point at all.
def V_exact(k):
    """V(k) = pi^(k/2) / Gamma(k/2 + 1) as an exact sympy expression."""
    return sy.pi**sy.Rational(k, 2) / sy.gamma(sy.Rational(k, 2) + 1)

def V_float(x):
    """The engine's floating-point implementation."""
    return fp.v_nball(x)

## D1 — The exact ladder

In [ ]:
print(f'{"n":>3} {"V(n) exact":>28} {"V(n) float":>22}')
for k in range(0, 17):
    e = V_exact(k)
    print(f'{k:>3} {str(e):>28} {V_float(k):>22.15f}')

## D2 — Step ratios along the Cayley–Dickson tower

ℝ→ℂ→ℍ→𝕆→𝕊 is `1→2→4→8→16`.

In [ ]:
tower = [1, 2, 4, 8, 16]
names = {1: 'R', 2: 'C', 4: 'H', 8: 'O', 16: 'S'}
print(f'{"step":>10} {"ratio exact":>16} {"ratio float":>22} {"== pi/2":>9}')
for lo, hi in zip(tower, tower[1:]):
    r_exact = sy.simplify(V_exact(hi)/V_exact(lo))
    r_float = V_float(hi)/V_float(lo)
    is_half_pi = sy.simplify(r_exact - sy.pi/2) == 0
    step = f'{names[lo]}->{names[hi]}'
    print(f'{step:>10} {str(r_exact):>16} {r_float:>22.15f} {str(is_half_pi):>9}')
print()
print('pi/2 =', repr(math.pi/2))
print()
print('The gain is constant for the first two steps and then is not.')

## D3 — The two-step recurrence

In [ ]:
print('V(n) = (2*pi/n) * V(n-2), checked symbolically:')
for k in (2, 4, 6, 8, 10, 12, 16, 24, 32):
    lhs = V_exact(k)
    rhs = sy.Rational(2, k)*sy.pi*V_exact(k-2)
    print(f'  n={k:>3}  exact difference = {sy.simplify(lhs-rhs)}')
print()
# The recurrence as an algorithm: V at even n from V(0)=1, no Gamma calls.
def V_by_recurrence(n_even):
    v = 1.0                      # V(0) = 1
    for k in range(2, n_even + 1, 2):
        v *= 2*math.pi/k
    return v

print(f'{"n":>3} {"by recurrence":>22} {"by gamma (engine)":>22} {"ulp diff":>10}')
for k in range(0, 17, 2):
    a, b = V_by_recurrence(k), V_float(k)
    d = 0.0 if a == b else (a - b)/math.ulp(b)
    print(f'{k:>3} {a:>22.15f} {b:>22.15f} {d:>10.2f}')

## D4 — Where the float implementation actually lands

`ulp` is the spacing between adjacent float64 values at that magnitude. A
difference of 1 ulp is the smallest representable disagreement; reporting
"agrees to 15 decimals" hides whether the answer was correctly rounded.

In [ ]:
def ulp_diff(got, exact):
    return 0.0 if got == exact else (got - exact)/math.ulp(exact)

print(f'{"n":>3} {"exact->float64":>22} {"engine":>22} {"ulp":>7}')
worst = 0.0
for k in range(0, 17):
    exact = float(V_exact(k))
    got = V_float(k)
    d = ulp_diff(got, exact)
    worst = max(worst, abs(d))
    print(f'{k:>3} {exact:>22.15f} {got:>22.15f} {d:>7.2f}')
print()
print(f'worst |ulp| over n=0..16 : {worst:.2f}')

In [ ]:
# The identity is exact. Is it exact in float64?
r21 = V_float(2)/V_float(1)
r42 = V_float(4)/V_float(2)
half_pi = math.pi/2
print(f'V(2)/V(1) float = {r21!r}')
print(f'V(4)/V(2) float = {r42!r}')
print(f'pi/2            = {half_pi!r}')
print()
print(f'V(2)/V(1) == pi/2 : {r21 == half_pi}   ({ulp_diff(r21, half_pi):+.1f} ulp)')
print(f'V(4)/V(2) == pi/2 : {r42 == half_pi}   ({ulp_diff(r42, half_pi):+.1f} ulp)')
print(f'the two ratios equal each other : {r21 == r42}')
print()
print('An exact identity that does not survive float64 evaluation. This is')
print('why P1 is tested symbolically and not by comparing doubles.')

## D5 — Locating n*

`V` is maximised where `dV/dn = 0`. Differentiating
`ln V = (n/2)ln π − ln Γ(n/2+1)` gives

```
ln(pi) = psi(n/2 + 1)
```

so `n*` is a root of `ψ(n/2+1) − ln π`, and `ψ` is monotone on `n > 0`, so the
root is unique and bracketable.

In [ ]:
LN_PI = math.log(math.pi)
calls = {'n': 0}

def f(x):
    calls['n'] += 1
    return ss.digamma(x/2 + 1) - LN_PI

print(f'f(1)  = {f(1.0):+.6f}')
print(f'f(20) = {f(20.0):+.6f}   -> sign change, root is bracketed')
calls['n'] = 0
root = so.brentq(f, 1.0, 20.0, xtol=1e-15, rtol=8.9e-16)
n_calls = calls['n']
print()
print(f'brentq root      : {root!r}')
print(f'digamma calls    : {n_calls}')
print(f'residual         : {abs(ss.digamma(root/2+1) - LN_PI):.3e}')
print(f'V(n*)            : {V_float(root)!r}')

In [ ]:
# The engine's grid search, for comparison.
peak = fp.v_nball_peak()
grid_n = peak['n_star']
print('engine v_nball_peak():')
for k, v in peak.items():
    print(f'   {k:>14} : {v!r}')
print()
grid_pts = 10000
grid_lo, grid_hi = 0.1, 20.0
spacing = (grid_hi - grid_lo)/(grid_pts - 1)
print(f'grid: {grid_pts} points over [{grid_lo}, {grid_hi}], spacing {spacing:.3e}')
print(f'digamma calls    : {grid_pts}')
print()
print(f'{"source":>22} {"n*":>22} {"error vs brentq":>18}')
for label, val in [('brentq (root find)', root),
                   ('engine grid search', grid_n),
                   ('stored N_STAR', fp.N_STAR)]:
    print(f'{label:>22} {val:>22.15f} {abs(val-root):>18.3e}')